# Web Traffic Forecasting

**Project 7 of 10** — Time Series Forecasting Portfolio

## Project Overview

This notebook forecasts **daily page views** for Wikipedia articles using the Kaggle *Web Traffic Time Series Forecasting* competition dataset. We select a representative subset of pages, aggregate to total daily views, and apply MLForecast models.

| Attribute | Value |
|-----------|-------|
| **Project type** | Time Series Forecasting |
| **Target variable** | `views` (daily page views) |
| **Date column** | date columns across CSV |
| **Frequency** | Daily (`D`) |
| **Seasonal period** | 7 (weekly cycle) |
| **Primary TS library** | MLForecast |
| **Kaggle competition** | `web-traffic-time-series-forecasting` |

## Learning Objectives

1. Download and parse a **wide-format** time-series dataset (pages as rows, dates as columns)
2. Reshape from wide to long format for analysis
3. Handle **missing page-view data** (NaN days)
4. Aggregate across pages for a total daily traffic signal
5. Explore **day-of-week effects** and trend in web traffic
6. Build naive and seasonal-naive baselines
7. Benchmark via LazyPredict on lag features
8. Run FLAML AutoML
9. Train MLForecast models (LightGBM, XGBoost, Ridge)
10. Evaluate with MAE, RMSE, MAPE

## Problem Statement

Given ~550 days of daily page views for ~145K Wikipedia articles, **forecast total daily page views for the next 14 days**.

Web traffic is inherently noisy with weekly seasonality (weekday/weekend patterns), trending topics, and occasional viral spikes.

## Why This Project Matters

- **Infrastructure planning**: CDN and server capacity depends on traffic forecasts.
- **Content strategy**: Predicting which pages will trend helps editorial teams prepare.
- **Advertising revenue**: Page view forecasts drive ad inventory projections.
- **Anomaly detection**: Deviations from forecast signal bot activity or viral events.

## Dataset Overview

| File | Description |
|------|-------------|
| `train_1.csv` | ~145K pages × ~550 days of daily views (wide format) |

### Format
- **Rows**: Wikipedia page names (with language, access type, agent info encoded in the name)
- **Columns**: Dates (YYYY-MM-DD)
- **Values**: Integer page views (many NaN for pages not yet created)

## Dataset Source & License Notes

- **Kaggle competition**: [Web Traffic Time Series Forecasting](https://www.kaggle.com/competitions/web-traffic-time-series-forecasting)
- **License**: Competition rules
- **Provider**: Google (via Kaggle)
- **Usage**: Educational purposes only

## Environment Setup

In [ ]:
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in [
    "kagglehub", "pandas", "numpy", "matplotlib", "seaborn", "plotly",
    "scikit-learn", "lazypredict", "flaml", "mlforecast", "lightgbm", "xgboost",
    "statsmodels", "scipy", "window-ops",
]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        _install(pkg)

print("All packages ready.")

## Imports

In [ ]:
import warnings, os, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge

from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

from mlforecast import MLForecast
import lightgbm as lgb
import xgboost as xgb
from window_ops.rolling import rolling_mean, rolling_std

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")

print("All imports successful.")

## Configuration & Constants

In [ ]:
PROJECT_NAME = "Web Traffic Forecasting"
KAGGLE_SLUG  = "web-traffic-time-series-forecasting"

TARGET  = "views"
FREQ    = "D"

SEASON_LENGTH    = 7
FORECAST_HORIZON = 14
TEST_SIZE        = FORECAST_HORIZON
VAL_SIZE         = FORECAST_HORIZON
RANDOM_STATE     = 42
FLAML_BUDGET     = 120
TOP_N_PAGES      = 500  # sample pages for tractability

print(f"Project : {PROJECT_NAME}")
print(f"Target  : {TARGET}  |  Freq: {FREQ}  |  Season: {SEASON_LENGTH}")
print(f"Horizon : {FORECAST_HORIZON} days")

## Kaggle Credential Check

In [ ]:
kaggle_ok = False

if os.environ.get("KAGGLE_USERNAME") or os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN"):
    print("Kaggle credentials found via environment variables.")
    kaggle_ok = True

kaggle_json = pathlib.Path.home() / ".kaggle" / "kaggle.json"
if kaggle_json.exists():
    print(f"Kaggle credentials found at {kaggle_json}")
    kaggle_ok = True

if not kaggle_ok:
    raise RuntimeError(
        "No Kaggle credentials found!\n"
        "Set KAGGLE_API_TOKEN env var, or place kaggle.json in ~/.kaggle/\n"
        "See: https://www.kaggle.com/settings -> Create New Token"
    )
print("Ready to download.")

## Dataset Download & Loading

The dataset is in wide format — pages as rows, dates as columns. We download, load, sample pages, and reshape to long format.

In [ ]:
import kagglehub

try:
    data_path = pathlib.Path(kagglehub.competition_download(KAGGLE_SLUG))
    print(f"Downloaded to: {data_path}")
except Exception as e:
    print(f"kagglehub download failed: {e}")
    os.makedirs("data", exist_ok=True)
    ret = os.system(f"kaggle competitions download -c {KAGGLE_SLUG} -p data/ --unzip")
    if ret != 0:
        raise RuntimeError("Download failed. Accept competition rules at kaggle.com first.")
    data_path = pathlib.Path("data")

csv_files = sorted(data_path.rglob("*.csv"))
for f in csv_files:
    print(f"  {f.name:40s}  {f.stat().st_size / 1e6:7.2f} MB")
assert len(csv_files) > 0, "No CSV files found!"

In [ ]:
# Load train_1.csv (wide format)
train_file = [f for f in csv_files if "train_1" in f.name.lower() or "train" in f.name.lower()][0]
raw = pd.read_csv(train_file)
print(f"Raw shape: {raw.shape} ({raw.shape[0]} pages x {raw.shape[1]-1} days)")
raw.head(3)

## Data Validation Checks

In [ ]:
print("=" * 60)
print("DATA VALIDATION REPORT")
print("=" * 60)
page_col = raw.columns[0]  # first column is page name
date_cols = raw.columns[1:]
print(f"  Pages          : {raw.shape[0]:,}")
print(f"  Date columns   : {len(date_cols)} ({date_cols[0]} to {date_cols[-1]})")
print(f"  Total NaN      : {raw[date_cols].isna().sum().sum():,} ({raw[date_cols].isna().mean().mean()*100:.1f}%)")
print(f"  All-NaN pages  : {raw[date_cols].isna().all(axis=1).sum()}")
print(f"  Negative values: {(raw[date_cols] < 0).sum().sum()}")
print("\nValidation complete.")

## Data Cleaning / Preprocessing

We sample the top pages by total views, aggregate to daily totals, and fill missing dates.

In [ ]:
# Sample top pages for tractability
page_col = raw.columns[0]
date_cols = raw.columns[1:]
raw["total_views"] = raw[date_cols].sum(axis=1)
sampled = raw.nlargest(TOP_N_PAGES, "total_views").copy()
print(f"Sampled {TOP_N_PAGES} pages (top by total views)")

# Aggregate to daily total views
daily_views = sampled[date_cols].sum(axis=0).reset_index()
daily_views.columns = ["date", "views"]
daily_views["date"] = pd.to_datetime(daily_views["date"])
daily_views = daily_views.sort_values("date").reset_index(drop=True)

# Fill missing dates
full_range = pd.date_range(daily_views["date"].min(), daily_views["date"].max(), freq="D")
daily_views = daily_views.set_index("date").reindex(full_range).rename_axis("date").reset_index()
daily_views["views"] = daily_views["views"].interpolate(method="linear")

print(f"Daily series: {len(daily_views)} days")
print(f"Date range: {daily_views['date'].min().date()} to {daily_views['date'].max().date()}")
daily_views.head()

## Exploratory Data Analysis

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=daily_views["date"], y=daily_views["views"],
                         name="Daily Views", line=dict(width=1)))
fig.add_trace(go.Scatter(x=daily_views["date"],
                         y=daily_views["views"].rolling(7).mean(),
                         name="7-day MA", line=dict(color="red", width=2)))
fig.update_layout(title="Wikipedia — Total Daily Page Views (Top 500 Pages)",
                  template="plotly_white")
fig.show()

In [ ]:
daily_views["dow"] = daily_views["date"].dt.day_name()
fig = px.box(daily_views, x="dow", y="views",
             category_orders={"dow": ["Monday","Tuesday","Wednesday","Thursday",
                                       "Friday","Saturday","Sunday"]},
             title="Page Views by Day of Week")
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
ts = daily_views.set_index("date")["views"].asfreq("D")
ts = ts.interpolate() if ts.isna().any() else ts
decomp = seasonal_decompose(ts, model="additive", period=SEASON_LENGTH)
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp.observed.plot(ax=axes[0], title="Observed")
decomp.trend.plot(ax=axes[1], title="Trend")
decomp.seasonal.plot(ax=axes[2], title="Weekly Seasonal")
decomp.resid.plot(ax=axes[3], title="Residual")
plt.tight_layout(); plt.show()

In [ ]:
adf = adfuller(ts.dropna())
print(f"ADF: stat={adf[0]:.4f} p={adf[1]:.6f} -> {'STATIONARY' if adf[1]<0.05 else 'NON-STATIONARY'}")

## Target Analysis

In [ ]:
print("Target Statistics:")
print(daily_views["views"].describe().to_string())
print(f"\nSkewness: {daily_views['views'].skew():.3f}")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(daily_views["views"], bins=30, edgecolor="black", alpha=0.7)
axes[0].set_title("Distribution")
axes[1].boxplot(daily_views["views"].dropna())
axes[1].set_title("Box Plot")
pd.plotting.lag_plot(daily_views["views"], lag=7, ax=axes[2])
axes[2].set_title("Lag-7 Plot")
plt.tight_layout(); plt.show()

## Train / Validation / Test Split

In [ ]:
ts_df = daily_views[["date", "views"]].rename(columns={"date":"ds","views":"y"}).copy()
n = len(ts_df)
ts_train = ts_df.iloc[:n-TEST_SIZE-VAL_SIZE].copy()
ts_val   = ts_df.iloc[n-TEST_SIZE-VAL_SIZE:n-TEST_SIZE].copy()
ts_test  = ts_df.iloc[n-TEST_SIZE:].copy()
print(f"Train:{len(ts_train)} Val:{len(ts_val)} Test:{len(ts_test)}")
assert ts_train["ds"].max() < ts_val["ds"].min()
assert ts_val["ds"].max() < ts_test["ds"].min()
print("No temporal overlap.")
ts_trainval = pd.concat([ts_train, ts_val]).reset_index(drop=True)

fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_train["ds"],y=ts_train["y"],name="Train"))
fig.add_trace(go.Scatter(x=ts_val["ds"],y=ts_val["y"],name="Val",line=dict(color="orange")))
fig.add_trace(go.Scatter(x=ts_test["ds"],y=ts_test["y"],name="Test",line=dict(color="red")))
fig.update_layout(title="Temporal Split", template="plotly_white")
fig.show()

## Feature Engineering

In [ ]:
def make_features(df):
    out = df.copy()
    for lag in [1, 7, 14, 21, 28]:
        out[f"lag_{lag}"] = out["y"].shift(lag)
    for w in [7, 14, 28]:
        out[f"rmean_{w}"] = out["y"].shift(1).rolling(w).mean()
        out[f"rstd_{w}"]  = out["y"].shift(1).rolling(w).std()
    out["dow"] = out["ds"].dt.dayofweek
    out["month"] = out["ds"].dt.month
    out["day"] = out["ds"].dt.day
    return out

feat = make_features(ts_df).dropna().reset_index(drop=True)
fcols = [c for c in feat.columns if c not in ["ds","y"]]
tc=ts_train["ds"].max(); vc=ts_val["ds"].max()
ft=feat[feat["ds"]<=tc]; fv=feat[(feat["ds"]>tc)&(feat["ds"]<=vc)]
fte=feat[feat["ds"]>vc]
X_tr,y_tr=ft[fcols],ft["y"]; X_v,y_v=fv[fcols],fv["y"]
X_te,y_te=fte[fcols],fte["y"]
X_tv=pd.concat([X_tr,X_v]); y_tv=pd.concat([y_tr,y_v])
print(f"X_train:{X_tr.shape} X_val:{X_v.shape} X_test:{X_te.shape}")

## Baseline Approaches

In [ ]:
def calc_metrics(actual, predicted, name="Model"):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mask = actual != 0
    mape = np.mean(np.abs((actual[mask]-predicted[mask])/actual[mask]))*100 if mask.any() else np.nan
    return {"Model":name,"MAE":round(mae,2),"RMSE":round(rmse,2),"MAPE":round(mape,2)}

results = []
actual_test = ts_test["y"].values
results.append(calc_metrics(pd.Series(actual_test),
    pd.Series(np.full(TEST_SIZE, ts_trainval["y"].iloc[-1])), "Naive"))
sn = ts_trainval["y"].iloc[-SEASON_LENGTH:].values
results.append(calc_metrics(pd.Series(actual_test),
    pd.Series(np.tile(sn,3)[:TEST_SIZE]), "Seasonal Naive (7)"))
results.append(calc_metrics(pd.Series(actual_test),
    pd.Series(np.full(TEST_SIZE, ts_trainval["y"].iloc[-7:].mean())), "MA(7)"))
print("Baselines:")
print(pd.DataFrame(results).to_string(index=False))

## LazyPredict Benchmark (Lag-Feature Tabular View)

LazyPredict on lag features — **not** native time-series forecasting.

In [ ]:
try:
    lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=RANDOM_STATE)
    lm, _ = lazy.fit(X_tr, X_v, y_tr, y_v)
    print("LazyPredict top 15:")
    print(lm.head(15).to_string())
    for i,(nm,row) in enumerate(lm.head(3).iterrows()):
        results.append({"Model":f"LP:{nm}","MAE":round(row.get("MAE",0),2),
                        "RMSE":round(row.get("RMSE",0),2),"MAPE":np.nan})
except Exception as e:
    print(f"LazyPredict failed: {e}")

## FLAML AutoML

In [ ]:
try:
    fl = AutoML()
    fl.fit(X_train=X_tv, y_train=y_tv, task="regression",
           time_budget=FLAML_BUDGET, metric="rmse", verbose=0, seed=RANDOM_STATE)
    fp = fl.predict(X_te)
    results.append(calc_metrics(pd.Series(actual_test[:len(fp)]),
                                pd.Series(fp), f"FLAML ({fl.best_estimator})"))
    print(f"FLAML best: {fl.best_estimator}")
except Exception as e:
    print(f"FLAML failed: {e}")

## MLForecast — Dedicated Time-Series Models

**Why MLForecast?** Web traffic forecasting involves daily data with strong weekly seasonality — exactly where gradient boosting with lag features excels. MLForecast automates lag/rolling feature creation and handles the forecasting loop natively.

In [ ]:
mlf_tv = ts_trainval.copy()
mlf_tv["unique_id"] = "total_views"

mlf = MLForecast(
    models={
        "LightGBM": lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05,
                                       num_leaves=31, random_state=RANDOM_STATE, verbosity=-1),
        "XGBoost": xgb.XGBRegressor(n_estimators=500, learning_rate=0.05,
                                     max_depth=6, random_state=RANDOM_STATE, verbosity=0),
        "Ridge": Ridge(alpha=1.0),
    },
    freq="D",
    lags=[1, 7, 14, 21, 28],
    lag_transforms={
        1: [(rolling_mean, 7), (rolling_mean, 14), (rolling_std, 7)],
        7: [(rolling_mean, 4)],
    },
    date_features=["dayofweek", "month", "day"],
)
mlf.fit(mlf_tv)
mlf_preds = mlf.predict(h=FORECAST_HORIZON)
print(f"MLForecast output: {mlf_preds.shape}")
mlf_preds.head()

In [ ]:
for mn in ["LightGBM", "XGBoost", "Ridge"]:
    if mn in mlf_preds.columns:
        pv = mlf_preds[mn].values[:TEST_SIZE]
        r = calc_metrics(pd.Series(actual_test), pd.Series(pv), f"MLF:{mn}")
        results.append(r)
        print(f"{mn}: MAE={r['MAE']}, RMSE={r['RMSE']}, MAPE={r['MAPE']}%")

## Top 3 Model Selection

In [ ]:
results_df = pd.DataFrame(results).dropna(subset=["RMSE"]).sort_values("RMSE")
print("All Models Ranked:")
print(results_df.to_string(index=False))
top3 = results_df.head(3)
print(f"\n{'='*50}\nTOP 3:\n{'='*50}")
print(top3.to_string(index=False))
fig = px.bar(results_df, x="Model", y="RMSE", title="Model Comparison — RMSE",
             color="RMSE", color_continuous_scale="RdYlGn_r")
fig.update_layout(template="plotly_white", xaxis_tickangle=-45)
fig.show()

## Final Evaluation — Forecast vs Actual

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_test["ds"], y=actual_test, name="Actual",
                         line=dict(color="black", width=3)))
for mn in ["LightGBM","XGBoost","Ridge"]:
    if mn in mlf_preds.columns:
        fig.add_trace(go.Scatter(x=ts_test["ds"], y=mlf_preds[mn].values[:TEST_SIZE],
                                 name=f"MLF:{mn}", line=dict(dash="dash")))
fig.update_layout(title="Forecast vs Actual — Test Period", template="plotly_white")
fig.show()

## Error Analysis

In [ ]:
best_col = "LightGBM" if "LightGBM" in mlf_preds.columns else mlf_preds.columns[-1]
best_pred = mlf_preds[best_col].values[:TEST_SIZE]
errors = actual_test - best_pred
pct_err = np.where(actual_test!=0, errors/actual_test*100, 0)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(errors, bins=15, edgecolor="black", alpha=0.7)
axes[0].set_title("Error Distribution"); axes[0].axvline(0, color="red", ls="--")
axes[1].plot(range(1,TEST_SIZE+1), np.abs(pct_err), marker="o")
axes[1].set_title("|% Error| by Day")
axes[2].scatter(actual_test, best_pred, alpha=0.7)
mn_,mx_ = min(actual_test.min(),best_pred.min()), max(actual_test.max(),best_pred.max())
axes[2].plot([mn_,mx_],[mn_,mx_],"r--")
axes[2].set_title("Actual vs Predicted")
plt.tight_layout(); plt.show()
print(f"Bias: {errors.mean():,.0f}  MAE: {np.abs(errors).mean():,.0f}")

## Interpretation & Insights

1. **Weekly cycle**: Wikipedia traffic clearly dips on weekends as work/school-related lookups decrease.
2. **Trending events**: Spikes correspond to news events driving article views.
3. **MLForecast captures weekly patterns**: Lag-7 and day-of-week features are the strongest predictors.
4. **Aggregation smooths noise**: Individual page forecasts are much noisier than the aggregate.

## Limitations

1. **Aggregated view**: We forecast total views across sampled pages; individual page forecasts need panel modeling.
2. **Page sampling**: Only top 500 pages by total views — misses long-tail.
3. **No external events**: News events, holidays drive traffic but are not included.
4. **Bot traffic**: Some views may be automated, distorting patterns.
5. **Single split**: Rolling-origin CV would give more robust estimates.

## How to Improve This Project

1. **Panel forecasting**: Use `unique_id` per page for per-article forecasts.
2. **External regressors**: Add holiday calendars, trending topics.
3. **Log transform**: `np.log1p()` on views to handle extreme spikes.
4. **Rolling CV**: Multiple temporal windows for evaluation.
5. **Language-specific models**: Train separate models for en/de/fr/ja Wikipedia.

## Production Considerations

1. **Daily retraining**: New traffic data arrives continuously.
2. **CDN scaling**: Use forecasts to pre-provision server capacity.
3. **Anomaly alerts**: Deviations from forecast signal bot attacks or viral events.
4. **Caching strategy**: High-forecast pages get priority caching.
5. **A/B testing**: Compare forecast-driven provisioning vs reactive scaling.

## Common Mistakes to Avoid

1. **Including bot traffic**: Bots create artificial patterns that mislead models.
2. **Ignoring weekday/weekend**: The strongest signal in web traffic.
3. **Random splits**: Always temporal for time series.
4. **MAPE with zero views**: Some pages have zero views — use MAE or sMAPE.
5. **Over-aggregation**: Monthly totals lose the weekly signal.

## Mini Challenge / Exercises

1. **Per-page forecasting**: Run MLForecast with `unique_id` per page.
2. **Language comparison**: Compare en.wikipedia vs ja.wikipedia traffic patterns.
3. **Holiday features**: Add country-specific holidays.
4. **Ensemble**: Average LightGBM + XGBoost. Better?
5. **28-day horizon**: Extend forecast. How does accuracy degrade?

## Final Summary & Key Takeaways

### What We Did
- Downloaded the **Web Traffic Time Series** competition data from Kaggle
- Parsed wide-format data, sampled top pages, aggregated to daily totals
- Explored weekly seasonality and trend patterns
- Built baselines (naive, seasonal naive, moving average)
- Benchmarked via LazyPredict and FLAML on lag features
- Trained MLForecast models (LightGBM, XGBoost, Ridge)
- Ranked all models and analyzed errors

### Key Takeaways
1. **Weekly seasonality dominates** web traffic — weekday/weekend pattern is the key signal
2. **MLForecast with gradient boosting** effectively captures both weekly cycles and trend
3. **Aggregation across pages** produces a smoother, more forecastable signal
4. **Per-page panel forecasting** is the natural production extension

---
*Notebook generated as part of a time-series forecasting portfolio.*
*For educational purposes only.*